In [1]:
import pandas as pd

In [5]:
# File paths
kpi_file = "OC_Transpo_Service_and_Ridership_KPIs.xlsx"
ops_file = "BusServiceActionPlanKPIs.xlsx"

In [7]:
# Load all sheets
kpi_sheets = pd.read_excel(kpi_file, sheet_name=None)
ops_sheets = pd.read_excel(ops_file, sheet_name=None)

In [9]:
# Check sheet names
print("KPI Sheets:")
print(kpi_sheets.keys())

KPI Sheets:
dict_keys(['01_otp_conventional', '02_otp_para_transpo', '03_ridership', '04_ridership_paratranspo', '05_otrain_service_delivery', '06_bus_service_delivery'])


In [11]:
print("\nOperations Sheets:")
print(ops_sheets.keys())


Operations Sheets:
dict_keys(['01_bus_service_delivery', '02_undelivered_trips', '03_reasons_for_undelivered', '04_fleet_health', '05_electric_bus_procurement'])


In [13]:
# Preview each KPI sheet
for name, df in kpi_sheets.items():
    print(f"\n===== {name} =====")
    print(df.head())


===== 01_otp_conventional =====
    month  target  regularity  punctuality
0  Jan 25      85          82         75.5
1  Feb 25      85          78         71.8
2  Mar 25      85          82         75.0
3  Apr 25      85          82         75.4
4  May 25      85          83         69.7

===== 02_otp_para_transpo =====
    month  overall
0  Jan 25       94
1  Feb 25       92
2  Mar 25       95
3  Apr 25       94
4  May 25       95

===== 03_ridership =====
    month  ridership
0  Jan 25        6.5
1  Feb 25        5.9
2  Mar 25        6.6
3  Apr 25        5.9
4  May 25        5.0

===== 04_ridership_paratranspo =====
    month  paratranspo-ridership Unnamed: 2
0  Jan 25                 72.049        NaN
1  Feb 25                 65.555           
2  Mar 25                 73.811        NaN
3  Apr 25                 75.477        NaN
4  May 25                 77.780        NaN

===== 05_otrain_service_delivery =====
    month  target  o-train-line-1  o-train-line-2-4
0  Jan 25    99.

In [39]:
#Test cleaning logic on one work sheet
# Pick one worksheet/tab from the KPI workbook
sheet_name = list(kpi_sheets.keys())[0]
df = kpi_sheets['01_otp_conventional'].copy()

# Drop empty unnamed columns
df = df.loc[:, ~df.columns.str.contains(r'^Unnamed', na=False)]

# Drop fully empty rows
df = df.dropna(how='all')

# Rename first column to Month
df = df.rename(columns={df.columns[0]: 'Month'})

# Convert Month to proper date
df['Date'] = pd.to_datetime(df['Month'], format='%b %y')

# Drop Month after creating Date
df = df.drop(columns=['Month'])

# Identify KPI columns (exclude Target)
kpi_columns = [col for col in df.columns if col.strip().lower() not in ['date', 'target']]

# Reshape from wide to long
df_long = df.melt(id_vars='Date', value_vars=kpi_columns, var_name='Original_Metric', value_name='Value')

# Standardize KPI names
kpi_name_map = {
    'regularity': 'Regularity',
    'punctuality': 'Punctuality',
    'overall': 'OTP ParaTranspo',
    'ridership': 'Ridership',
    'paratranspo-ridership': 'Para Ridership',
    'O-train-line-1': 'OTrain Line 1',
    'O-train-line-2–4': 'OTrain Line 2-4',
    'Bus Service Delivery': 'Bus Service Delivery'
}

df_long['KPI_Name'] = df_long['Original_Metric'].replace(kpi_name_map)

df_long.head()

,Date,Original_Metric,Value,KPI_Name
0,2025-01-01,regularity,82.0,Regularity
1,2025-02-01,regularity,78.0,Regularity
2,2025-03-01,regularity,82.0,Regularity
3,2025-04-01,regularity,82.0,Regularity
4,2025-05-01,regularity,83.0,Regularity


In [41]:
#drop duplicates
df_long[['Original_Metric', 'KPI_Name']].drop_duplicates()

,Original_Metric,KPI_Name
0,regularity,Regularity
14,punctuality,Punctuality


In [43]:
metadata_map = {
    'Regularity': {'Target': 85, 'Category': 'Reliability', 'Mode': 'Bus'},
    'Punctuality': {'Target': 85, 'Category': 'Reliability', 'Mode': 'Bus'},
    'OTP ParaTranspo': {'Target': 85, 'Category': 'Equity', 'Mode': 'Para'},
    'Ridership': {'Target': None, 'Category': 'Ridership', 'Mode': 'Bus'},
    'Para Ridership': {'Target': None, 'Category': 'Equity', 'Mode': 'Para'},
    'OTrain Line 1': {'Target': None, 'Category': 'Reliability', 'Mode': 'Train'},
    'OTrain Line 2-4': {'Target': None, 'Category': 'Reliability', 'Mode': 'Train'},
    'Bus Service Delivery': {'Target': 99.5, 'Category': 'Reliability', 'Mode': 'Bus'}
}

meta_df = pd.DataFrame.from_dict(metadata_map, orient='index').reset_index()
meta_df = meta_df.rename(columns={'index': 'KPI_Name'})

df_long = df_long.merge(meta_df, on='KPI_Name', how='left')

df_long.head()

,Date,Original_Metric,Value,KPI_Name,Target,Category,Mode
0,2025-01-01,regularity,82.0,Regularity,85.0,Reliability,Bus
1,2025-02-01,regularity,78.0,Regularity,85.0,Reliability,Bus
2,2025-03-01,regularity,82.0,Regularity,85.0,Reliability,Bus
3,2025-04-01,regularity,82.0,Regularity,85.0,Reliability,Bus
4,2025-05-01,regularity,83.0,Regularity,85.0,Reliability,Bus


In [47]:
def clean_kpi_sheet(df):
    
    df = df.copy()
    
    # Drop unnamed columns
    df = df.loc[:, ~df.columns.str.contains(r'^Unnamed', na=False)]

    # Drop empty rows
    df = df.dropna(how='all')

    # Rename Month column
    df = df.rename(columns={df.columns[0]: 'Month'})

    # Convert to date
    df['Date'] = pd.to_datetime(df['Month'], format='%b %y')

    # Drop Month
    df = df.drop(columns=['Month'])

    # Exclude target column
    kpi_columns = [
        col for col in df.columns
        if col.strip().lower() not in ['date', 'target']
    ]

    # Melt
    df_long = df.melt(
        id_vars='Date',
        value_vars=kpi_columns,
        var_name='Original_Metric',
        value_name='Value'
    )

    # Standardize KPI names (use your working mapping)
    kpi_name_map = {
        'regularity': 'Regularity',
        'punctuality': 'Punctuality',
        'overall': 'OTP ParaTranspo',
        'ridership': 'Ridership',
        'para ridership': 'Para Ridership',
        'o-train line 1': 'OTrain Line 1',
        'o-train line 2–4': 'OTrain Line 2-4',
        'bus service delivery': 'Bus Service Delivery'
    }

    df_long['KPI_Name'] = df_long['Original_Metric'].replace(kpi_name_map)

    # Metadata
    metadata_map = {
        'Regularity': {'Target': 85, 'Category': 'Reliability', 'Mode': 'Bus'},
        'Punctuality': {'Target': 85, 'Category': 'Reliability', 'Mode': 'Bus'},
        'OTP ParaTranspo': {'Target': 85, 'Category': 'Equity', 'Mode': 'Para'},
        'Ridership': {'Target': None, 'Category': 'Ridership', 'Mode': 'Bus'},
        'Para Ridership': {'Target': None, 'Category': 'Equity', 'Mode': 'Para'},
        'OTrain Line 1': {'Target': None, 'Category': 'Reliability', 'Mode': 'Train'},
        'OTrain Line 2-4': {'Target': None, 'Category': 'Reliability', 'Mode': 'Train'},
        'Bus Service Delivery': {'Target': 99.5, 'Category': 'Reliability', 'Mode': 'Bus'}
    }

    meta_df = pd.DataFrame.from_dict(metadata_map, orient='index').reset_index()
    meta_df = meta_df.rename(columns={'index': 'KPI_Name'})

    df_long = df_long.merge(meta_df, on='KPI_Name', how='left')

    return df_long

In [49]:
kpi_cleaned_list = []

for sheet_name, sheet_df in kpi_sheets.items():
    cleaned_df = clean_kpi_sheet(sheet_df)
    cleaned_df['Source_Sheet'] = sheet_name
    kpi_cleaned_list.append(cleaned_df)

fact_kpi_monthly = pd.concat(kpi_cleaned_list, ignore_index=True)

fact_kpi_monthly.head()

,Date,Original_Metric,Value,KPI_Name,Target,Category,Mode,Source_Sheet
0,2025-01-01,regularity,82.0,Regularity,85.0,Reliability,Bus,01_otp_conventional
1,2025-02-01,regularity,78.0,Regularity,85.0,Reliability,Bus,01_otp_conventional
2,2025-03-01,regularity,82.0,Regularity,85.0,Reliability,Bus,01_otp_conventional
3,2025-04-01,regularity,82.0,Regularity,85.0,Reliability,Bus,01_otp_conventional
4,2025-05-01,regularity,83.0,Regularity,85.0,Reliability,Bus,01_otp_conventional


In [51]:
for sheet_name, df in ops_sheets.items():
    print(f"\n===== {sheet_name} =====")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head())


===== 01_bus_service_delivery =====
Shape: (63, 5)
Columns: ['day', 'target', 'bus_service_delivery', 'weekly_average', 'Unnamed: 4']


,day,target,bus_service_delivery,weekly_average,Unnamed: 4
0,Jan 4,99.5,99.4,94.1,
1,Jan 5,99.5,89.5,94.1,NaN
2,Jan 6,99.5,94.5,94.1,NaN
3,Jan 7,99.5,93.7,94.1,
4,Jan 8,99.5,93.9,94.1,NaN



===== 02_undelivered_trips =====
Shape: (63, 3)
Columns: ['day', 'undelivered_trips', 'weekly_average']


,day,undelivered_trips,weekly_average
0,Jan 4,29,395.428571
1,Jan 5,812,395.428571
2,Jan 6,421,395.428571
3,Jan 7,471,395.428571
4,Jan 8,448,395.428571



===== 03_reasons_for_undelivered =====
Shape: (36, 4)
Columns: ['week', 'reason', 'reason_percentage', 'reason_numbers']


,week,reason,reason_percentage,reason_numbers
0,January 4,Mechanical Breakdown,0.12,341
1,January 4,No Bus Available,0.46,1282
2,January 4,On Street Adjustment,0.27,739
3,January 4,Operator Availability,0.15,406
4,January 11,Mechanical Breakdown,0.16,366



===== 04_fleet_health =====
Shape: (63, 3)
Columns: ['day', 'average_bus_available', 'target_bus_requirements']


,day,average_bus_available,target_bus_requirements
0,Jan 4,-,-
1,Jan 5,465.5,520
2,Jan 6,466.5,520
3,Jan 7,462,520
4,Jan 8,462,520



===== 05_electric_bus_procurement =====
Shape: (4, 5)
Columns: ['quarter', 'quarter_fr', 'projected_zebs_deliveries', 'forecasted_zebs_available', 'actual_zebs_available']


,quarter,quarter_fr,projected_zebs_deliveries,forecasted_zebs_available,actual_zebs_available
0,Q1,T1,110,89,64
1,Q2,T2,158,150,-
2,Q3,T3,208,193,-
3,Q4,T4,242,237,-


In [53]:
ops_sheets['05_electric_bus_procurement'] = (
    ops_sheets['05_electric_bus_procurement']
    .drop(columns=['quarter_fr'])
)

In [55]:
ops_sheets['05_electric_bus_procurement'].columns

Index(['quarter', 'projected_zebs_deliveries', 'forecasted_zebs_available',
       'actual_zebs_available'],
      dtype='object')

In [57]:
df_ops = ops_sheets['01_bus_service_delivery'].copy()

# Drop unnamed columns
df_ops = df_ops.loc[:, ~df_ops.columns.str.contains(r'^Unnamed', na=False)]

# Drop empty rows
df_ops = df_ops.dropna(how='all')

# Preview
df_ops.head()

,day,target,bus_service_delivery,weekly_average
0,Jan 4,99.5,99.4,94.1
1,Jan 5,99.5,89.5,94.1
2,Jan 6,99.5,94.5,94.1
3,Jan 7,99.5,93.7,94.1
4,Jan 8,99.5,93.9,94.1


In [59]:
#Clean the sheets

In [63]:
df_ops = ops_sheets['01_bus_service_delivery'].copy()

# Drop unnamed columns
df_ops = df_ops.loc[:, ~df_ops.columns.str.contains(r'^Unnamed', na=False)]

# Drop empty rows
df_ops = df_ops.dropna(how='all')

# Rename columns
df_ops = df_ops.rename(columns={'day': 'Date'})

# Add year explicitly, then convert
df_ops['Date'] = pd.to_datetime(df_ops['Date'].astype(str) + ' 2026', format='%b %d %Y')

# Standardize column names
df_ops.columns = (
    df_ops.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)

df_ops.head()

,date,target,bus_service_delivery,weekly_average
0,2026-01-04,99.5,99.4,94.1
1,2026-01-05,99.5,89.5,94.1
2,2026-01-06,99.5,94.5,94.1
3,2026-01-07,99.5,93.7,94.1
4,2026-01-08,99.5,93.9,94.1


In [65]:
df_ops = df_ops.drop(columns=['target', 'weekly_average'])
df_ops.head()

,date,bus_service_delivery
0,2026-01-04,99.4
1,2026-01-05,89.5
2,2026-01-06,94.5
3,2026-01-07,93.7
4,2026-01-08,93.9


In [67]:
#Cleaning step for sheet 02_undelivered_trips

df_undelivered = ops_sheets['02_undelivered_trips'].copy()

# Drop unnamed columns
df_undelivered = df_undelivered.loc[:, ~df_undelivered.columns.str.contains(r'^Unnamed', na=False)]

# Drop empty rows
df_undelivered = df_undelivered.dropna(how='all')

df_undelivered.head()

,day,undelivered_trips,weekly_average
0,Jan 4,29,395.428571
1,Jan 5,812,395.428571
2,Jan 6,421,395.428571
3,Jan 7,471,395.428571
4,Jan 8,448,395.428571


In [69]:
#Clean the sheet
df_undelivered = ops_sheets['02_undelivered_trips'].copy()

# Drop unnamed columns
df_undelivered = df_undelivered.loc[:, ~df_undelivered.columns.str.contains(r'^Unnamed', na=False)]

# Drop empty rows
df_undelivered = df_undelivered.dropna(how='all')

# Rename day → Date
df_undelivered = df_undelivered.rename(columns={'day': 'Date'})

# Convert Date (add year)
df_undelivered['Date'] = pd.to_datetime(
    df_undelivered['Date'].astype(str) + ' 2026',
    format='%b %d %Y'
)

# Standardize column names
df_undelivered.columns = (
    df_undelivered.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)

# Drop weekly average (not needed)
df_undelivered = df_undelivered.drop(columns=['weekly_average'])

df_undelivered.head()

,date,undelivered_trips
0,2026-01-04,29
1,2026-01-05,812
2,2026-01-06,421
3,2026-01-07,471
4,2026-01-08,448


In [77]:
#cleaning step for 03_reasons_for_undelivered

df_reasons = ops_sheets['03_reasons_for_undelivered'].copy()

df_reasons = df_reasons.loc[:, ~df_reasons.columns.str.contains(r'^Unnamed', na=False)]

# Drop empty rows
df_reasons = df_reasons.dropna(how='all')

# Standardize column names
df_reasons.columns = (
    df_reasons.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)

df_reasons.head()

,week,reason,reason_percentage,reason_numbers
0,January 4,Mechanical Breakdown,0.12,341
1,January 4,No Bus Available,0.46,1282
2,January 4,On Street Adjustment,0.27,739
3,January 4,Operator Availability,0.15,406
4,January 11,Mechanical Breakdown,0.16,366


In [79]:
#convert week to date
df_reasons['date'] = pd.to_datetime(
    df_reasons['week'].astype(str) + ' 2026',
    format='%B %d %Y'
)

In [81]:
#drop original week column

df_reasons = df_reasons.drop(columns=['week'])

In [85]:
#cleaning step for 04_fleet_health

df_fleet = ops_sheets['04_fleet_health'].copy()

df_fleet = df_fleet.loc[:, ~df_fleet.columns.str.contains(r'^Unnamed', na=False)]
df_fleet = df_fleet.dropna(how='all')

df_fleet.head()

,day,average_bus_available,target_bus_requirements
0,Jan 4,-,-
1,Jan 5,465.5,520
2,Jan 6,466.5,520
3,Jan 7,462,520
4,Jan 8,462,520


In [87]:
df_fleet = ops_sheets['04_fleet_health'].copy()

# Drop unnamed columns
df_fleet = df_fleet.loc[:, ~df_fleet.columns.str.contains(r'^Unnamed', na=False)]

# Drop empty rows
df_fleet = df_fleet.dropna(how='all')

# Rename day → Date
df_fleet = df_fleet.rename(columns={'day': 'Date'})

# Replace '-' with NaN
df_fleet = df_fleet.replace('-', pd.NA)

# Convert Date (add year)
df_fleet['Date'] = pd.to_datetime(
    df_fleet['Date'].astype(str) + ' 2026',
    format='%b %d %Y'
)

# Standardize column names
df_fleet.columns = (
    df_fleet.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)

df_fleet.head()

,date,average_bus_available,target_bus_requirements
0,2026-01-04,<NA>,<NA>
1,2026-01-05,465.5,520
2,2026-01-06,466.5,520
3,2026-01-07,462,520
4,2026-01-08,462,520


In [89]:
#create the date table

date_range = pd.date_range(start='2025-01-01', end='2026-02-28', freq='D')

dim_date = pd.DataFrame({'date': date_range})

# Add useful columns
dim_date['year'] = dim_date['date'].dt.year
dim_date['month'] = dim_date['date'].dt.month
dim_date['month_name'] = dim_date['date'].dt.strftime('%B')
dim_date['year_month'] = dim_date['date'].dt.to_period('M').astype(str)
dim_date['quarter'] = dim_date['date'].dt.quarter
dim_date['day_of_week'] = dim_date['date'].dt.day_name()
dim_date['is_weekend'] = dim_date['day_of_week'].isin(['Saturday', 'Sunday'])

dim_date.head()

,date,year,month,month_name,year_month,quarter,day_of_week,is_weekend
0,2025-01-01,2025,1,January,2025-01,1,Wednesday,False
1,2025-01-02,2025,1,January,2025-01,1,Thursday,False
2,2025-01-03,2025,1,January,2025-01,1,Friday,False
3,2025-01-04,2025,1,January,2025-01,1,Saturday,True
4,2025-01-05,2025,1,January,2025-01,1,Sunday,True


In [91]:
#export the data

fact_kpi_monthly.to_csv("fact_kpi_monthly.csv", index=False)
df_ops.to_csv("fact_bus_service_delivery_daily.csv", index=False)
df_undelivered.to_csv("fact_undelivered_trips_daily.csv", index=False)
df_reasons.to_csv("fact_reasons_weekly.csv", index=False)
df_fleet.to_csv("fact_fleet_health_daily.csv", index=False)
dim_date.to_csv("dim_date.csv", index=False)

In [95]:
#cleaning step for

df_electric = ops_sheets['05_electric_bus_procurement'].copy()

# Drop French column 
df_electric = df_electric.drop(columns=[col for col in df_electric.columns if 'fr' in col.lower()])

# Standardize column names
df_electric.columns = (
    df_electric.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)

df_electric.head()

,quarter,projected_zebs_deliveries,forecasted_zebs_available,actual_zebs_available
0,Q1,110,89,64
1,Q2,158,150,-
2,Q3,208,193,-
3,Q4,242,237,-


In [97]:
#convert quarter to date

quarter_map = {
    'Q1': '2026-01-01',
    'Q2': '2026-04-01',
    'Q3': '2026-07-01',
    'Q4': '2026-10-01'
}

df_electric['date'] = pd.to_datetime(df_electric['quarter'].map(quarter_map))

In [99]:
df_electric.to_csv("fact_electric_bus_quarterly.csv", index=False)